# The perceptron trick in code

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=100, n_features=2, n_informative=1, n_redundant=0, n_classes=2,
                           n_clusters_per_class=1, random_state=41, hypercube=False, class_sep=10)
print(X.shape, np.bincount(y))

## The algorithm

In [ ]:
def step(z):
    return 1 if z > 0 else 0

def perceptron(X, y, lr=0.1, loops=1000, seed=0):
    rng = np.random.default_rng(seed)       # fixed seed: the same random picks every run
    X = np.insert(X, 0, 1, axis=1)          # column of 1s for the intercept
    weights = np.ones(X.shape[1])
    for _ in range(loops):
        j = rng.integers(0, X.shape[0])
        y_hat = step(X[j] @ weights)
        weights = weights + lr * (y[j] - y_hat) * X[j]
    return weights[0], weights[1:]

intercept_, coef_ = perceptron(X, y)
print("intercept", intercept_, "coefficients", coef_)

## From w0 + w1 x1 + w2 x2 = 0 to x2 = m x1 + b

In [ ]:
m = -coef_[0] / coef_[1]
b = -intercept_ / coef_[1]
print(f"x2 = {m:.2f} x1 + {b:.2f}")
Xb = np.insert(X, 0, 1, axis=1)
w = np.r_[intercept_, coef_]
print("misclassified:", ((Xb @ w > 0).astype(int) != y).sum())

## Five random orders vs logistic regression: distance from the line to the nearest point of each class

In [ ]:
def gaps(w):
    d = (Xb @ w) / np.linalg.norm(w[1:])
    return round(d[y == 1].min(), 3), round(-d[y == 0].max(), 3)

for seed in range(5):
    i, c = perceptron(X, y, seed=seed)
    print("perceptron seed", seed, gaps(np.r_[i, c]))
lr = LogisticRegression(C=100, max_iter=10000).fit(X, y)
print("logistic regression ", gaps(np.r_[lr.intercept_, lr.coef_[0]]))